Here we have implemented human in loop in 2ways

1.With Custom tools
2.Using Pre-defined libraries(easy to understand)

although the working is same

#### For some actions, you may want to require human approval before running to ensure that everything is running as intended.

In [ ]:
from typing import Annotated
import operator,json
from typing import TypedDict, Annotated, Sequence
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph,END,START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_KEY")
llm=ChatGroq(model_name="gemma2-9b-it")

In [ ]:
llm.invoke("hi").content

Here we are creating this function as a tool

In [ ]:
@tool
def multiply(first_number:int, second_number:int)->int:
    """multiply two integer number"""
    return first_number * second_number

In [ ]:
multiply({"first_number":24,"second_number":364})

In [ ]:
multiply.invoke({"first_number":24,"second_number":364})

In [ ]:
from langchain_tavily import TavilySearch
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_KEY")

In [ ]:
@tool
def search(query:str):
    """perform the web search on the user query"""
    tavily=TavilySearch(topic='general',max_results=5)
    result=tavily.invoke(query)
    return result

In [ ]:
search("who is a current president of USA?")

In [ ]:
search.invoke("who is a current president of USA?")

In [ ]:
tools=[search,multiply]

In [ ]:
model_with_tools = llm.bind_tools(tools)

In [ ]:
tool_mapping={tool.name: tool for tool in tools}

In [ ]:
tool_mapping

In [ ]:
response = model_with_tools.invoke("who is a current president of USA?")

In [ ]:
response

In [ ]:
tool_details=response.additional_kwargs.get("tool_calls")

In [ ]:
tool_details

In [ ]:
tool_details[0]["function"]["name"]

In [ ]:
tool_details[0]["function"]["arguments"]

In [ ]:
json.loads(tool_details[0]["function"]["arguments"])

In [ ]:
tool_mapping[tool_details[0]["function"]["name"]].invoke(json.loads(tool_details[0]["function"]["arguments"]))

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

In [ ]:
def invoke_model(state:AgentState):
    messages = state['messages']
    question = messages[-1]   ## Fetching the user question
    return {"messages":[model_with_tools.invoke(question)]}

Human in Loop

if tool required is search it will ask from user if they want to go ahead with the search tool

In [ ]:
def invoke_tool(state:AgentState):
    tool_details= state['messages'][-1].additional_kwargs.get("tool_calls", [])[0]
    
    if tool_details is None:
        raise Exception("no tool call found")
    
    print(f'Selected tool: {tool_details.get("function").get("name")}')
    
    if tool_details.get("function").get("name")=="search":
        response = input(prompt=f"[y/n] continue with expensive web search?") #human in loop
        if response == "n":
            raise Exception("web search discard")
        
    response = tool_mapping[tool_details['function']['name']].invoke(json.loads(tool_details.get("function").get("arguments")))
    return {"messages" : [response]}

In [ ]:
def router(state):
    tool_calls = state['messages'][-1].additional_kwargs.get("tool_calls", [])
    if len(tool_calls):
        return "tool"
    else:
        return "end"

In [ ]:
graph = StateGraph(AgentState) ### StateGraph with AgentState

graph.add_node("ai_assistant", invoke_model)

graph.add_node("tool", invoke_tool)

In [ ]:
graph.add_conditional_edges("ai_assistant", router, {"tool": "tool","end": END,})

graph.add_edge("tool", END)

#graph.add_edge("tool", "ai_assistant") #if you want to get the tool back to ai assistant get

graph.set_entry_point("ai_assistant")

In [ ]:
app = graph.compile()

In [ ]:
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
for s in app.stream({"messages": ["who is upcoming president of USA?"]}):
    print(list(s.values())[0])
    print("----")

In [ ]:
for s in app.stream({"messages": ["what is multiplication of 23 and 46?"]}):
    print(list(s.values())[0])
    print("----")

if we will give no here then it will show exception

and at the end will mention web search discard

In [ ]:
for s in app.stream({"messages": ["what is the total amount of money exist over the earth?"]}):
    print(list(s.values())[0])
    print("----")

#### LangGraph supports human-in-the-loop workflows in a number of ways. In this section, we will use LangGraph's interrupt_before functionality to always break the tool node.

In [ ]:
from langchain_groq import ChatGroq
llm=ChatGroq(model_name="Gemma2-9b-It")

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
tavily=TavilySearchResults()

In [ ]:
tools = [tavily]

In [ ]:
llm_with_tools = llm.bind_tools(tools)

In [ ]:
def ai_assistant(state: AgentState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

In [ ]:
memory = MemorySaver()

In [ ]:
graph_builder = StateGraph(AgentState)
graph_builder.add_node("ai_assistant", ai_assistant)

tool_node = ToolNode(tools=tools)
graph_builder.add_node("tools", tool_node)

In [ ]:
graph_builder.add_edge(START, "ai_assistant")

graph_builder.add_conditional_edges(
    "ai_assistant",
    tools_condition,
)
graph_builder.add_edge("tools", "ai_assistant")

In [ ]:
app2 = graph_builder.compile(
    checkpointer=memory, #just adding memory after each iteration
    # This is new!
    interrupt_before=["tools"], #before exectuing tools it will interupt and will need permission to execute further
    # Note: can also interrupt __after__ tools, if desired.
    # interrupt_after=["tools"]
)

In [ ]:
from IPython.display import Image, display
display(Image(app2.get_graph().draw_mermaid_png()))

In [ ]:
user_input = "what is current a capital of india?"
config = {"configurable": {"thread_id": "1"}}
# The config is the **second positional argument** to stream() or invoke()!
events = app2.stream(
    {"messages": [("user", user_input)]}, config, stream_mode="values"
)

Here it's not giving the answer to query as it stops before exectuing the tool

In [ ]:
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

In [ ]:
snapshot = app2.get_state(config)
snapshot.next  #the next process that was going to occur but stopped due to interupption

In [ ]:
last_message=snapshot.values["messages"][-1]
last_message.tool_calls

Now to allow it to work and not to interupt add None

In [ ]:
# `None` will append nothing new to the current state, letting it resume as if it had never been interrupted
events = app2.stream(None, config, stream_mode="values")

In [ ]:
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

In [ ]:
user_input = "what is a weather in new delhi?"

config = {"configurable": {"thread_id": "1"}}

In [ ]:
# The config is the **second positional argument** to stream() or invoke()!
events = app2.stream(
    {"messages": [("user", user_input)]}, config, stream_mode="values"
)

In [ ]:
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

In [ ]:
snapshot = app2.get_state(config)

In [ ]:
snapshot.next

In [ ]:
last_message=snapshot.values["messages"][-1]

In [ ]:
last_message.tool_calls

In [ ]:
# `None` will append nothing new to the current state, letting it resume as if it had never been interrupted
events = app2.stream(None, config, stream_mode="values")

In [ ]:
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

In [ ]:
app2.get_state(config)

In [ ]:
snapshot=app2.get_state(config)

In [ ]:
snapshot.next

In [ ]:
user_input = "recent news ?"

config = {"configurable": {"thread_id": "1"}}

In [ ]:
# The config is the **second positional argument** to stream() or invoke()!
events = app2.stream(
    {"messages": [("user", user_input)]}, config, stream_mode="values"
)

In [ ]:
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

In [ ]:
snapshot=app2.get_state(config)

In [ ]:
current_message = snapshot.values["messages"][-1]

In [ ]:
current_message.pretty_print()

In [ ]:
tool_call_id = current_message.tool_calls[0]["id"] 

In [ ]:
tool_call_id

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage

In [ ]:
answer = "it is just related to raining which is happing on daily basis"

In [ ]:
new_messages = [
    ToolMessage(content=answer, tool_call_id=tool_call_id),
    AIMessage(content=answer),
]

In [ ]:
app2.update_state(
    config,
    {"messages": new_messages},
)

In [ ]:
print(app2.get_state(config).values["messages"][-1:])